In [2]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline 
import pandas as pd
from scipy import stats
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, accuracy_score

In [3]:
reviews_df = pd.read_csv("final_yelp_dataset.csv")
reviews_df = reviews_df[reviews_df['tier'] != '$$$$']
reviews_df.head(5)

,business_id,name,cuisine,city,state,total_business_stars,review_count,categories,tier,user_id,individual_stars,text,date
0,YB26JvvGS2LgkxEKOObSAw,Unagi & Sushi,Japanese,Metairie,LA,4.0,62,"['Sushi Bars', 'Restaurants', 'Japanese']",$$,iAD32p6h32eKDVxsPHSRHA,5.0,i've been eating at this restaurant for over 5...,2021-01-08 01:49:36
1,jfIwOEXcVRyhZjM4ISOh4g,Oreland Pizza,American,Oreland,PA,3.0,29,"['Sandwiches', 'Restaurants', 'Burgers', 'Pizza']",$$,rYvWv-Ny16b1lMcw1IP7JQ,1.0,how does a delivery person from here get lost ...,2021-01-02 00:19:00
2,yE1raqkLX7OZsjmX3qKIKg,Butcher & Bee,American,Nashville,TN,4.0,863,"['Middle Eastern', 'American (New)', 'Restaura...",$$,j4qNLF-VNRF2DwBkUENW-w,5.0,two words: whipped. feta. \nexplosion of amazi...,2021-01-27 23:28:03
3,3PlpoDgeAQAreL8FM2LelA,Kauboi Izakaya,Japanese,Reno,NV,4.0,312,"['Sushi Bars', 'Pubs', 'Gastropubs', 'Japanese...",$$,h41RIr5Rtkq7EoJ0tU0wgQ,4.0,place was great as well as parking. \nfood was...,2021-01-30 03:15:20
4,Rv6P37KiiuowrXti2JHZNQ,Nuevo Vallarta Authentic Mexican Food,Mexican,Pinellas Park,FL,4.0,106,"['Event Planning & Services', 'Chicken Wings',...",$,8_yoGifxsLHLRPtyDgMxhw,5.0,got this to go and it is for sure authentic! t...,2021-03-11 17:17:46


In [4]:
# DistilBERT sentiment pipeline
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0  
)

# sample per cuisine and tier 
SAMPLE_PER_GROUP = 5000

sampled_df = (
    reviews_df.groupby(['cuisine', 'tier'], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_GROUP), random_state=42))
    .reset_index(drop=True)
)

print(sampled_df.groupby(['cuisine', 'tier']).size().unstack(fill_value=0))
print(f"\nTotal reviews: {len(sampled_df)}")

Device set to use mps:0


tier         $    $$   $$$
cuisine                   
American  5000  5000  5000
Chinese   3706  5000   177
French     157  1307   855
Greek      959  2582    98
Indian     300  4541    60
Italian   2950  5000  2769
Japanese   456  5000  1302
Korean     225  2098    63
Mexican   5000  5000   177
Thai       258  5000    88

Total reviews: 70128


In [5]:
# setup: need a labeled subset to validate against 
# Use star rating as proxy label: 4-5 stars = positive, 1-2 stars = negative, drop 3
labeled_df = sampled_df[sampled_df['individual_stars'] != 3].copy()
labeled_df['true_label'] = (labeled_df['individual_stars'] >= 4).astype(int)  # 1=positive, 0=negative

print(f"Labeled subset: {len(labeled_df)} reviews")

Labeled subset: 64297 reviews


In [6]:

# optimize max length
print("\n── Optimizing max_length ──")
max_length_options = [64, 128, 256, 512]

for max_len in max_length_options:
    preds = sentiment_model(
        labeled_df['text'].tolist()[:500],  # use 500 reviews for speed
        truncation=True,
        max_length=max_len,
        batch_size=128
    )
    pred_labels = [1 if r['label'] == 'POSITIVE' else 0 for r in preds]
    true_labels = labeled_df['true_label'].values[:500]
    f1 = f1_score(true_labels, pred_labels)
    acc = accuracy_score(true_labels, pred_labels)
    print(f"  max_length={max_len:4d}:  F1={f1:.4f},  Accuracy={acc:.4f}")




── Optimizing max_length ──
  max_length=  64:  F1=0.8954,  Accuracy=0.9000
  max_length= 128:  F1=0.9317,  Accuracy=0.9340
  max_length= 256:  F1=0.9333,  Accuracy=0.9360
  max_length= 512:  F1=0.9353,  Accuracy=0.9380


In [7]:
# optimize classification threshold with K-Fold CV 
print("\n── Optimizing classification threshold (5-fold CV) ──")

# Get raw scores for the labeled subset
raw_preds = sentiment_model(
    labeled_df['text'].tolist(),
    truncation=True,
    max_length=128,  # use best max_length from above
    batch_size=128
)

# Convert to positive probability
labeled_df['raw_positive_score'] = [
    r['score'] if r['label'] == 'POSITIVE' else 1 - r['score']
    for r in raw_preds
]

# K-Fold CV over thresholds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
thresholds = np.arange(0.3, 0.8, 0.05)
threshold_f1s = {t: [] for t in thresholds}

for train_idx, val_idx in kf.split(labeled_df):
    val = labeled_df.iloc[val_idx]
    for thresh in thresholds:
        preds = (val['raw_positive_score'] >= thresh).astype(int)
        f1 = f1_score(val['true_label'], preds)
        threshold_f1s[thresh].append(f1)

print(f"\n  {'Threshold':>10}  {'Mean F1':>10}  {'Std':>8}")
best_thresh, best_f1 = 0.5, 0
for thresh, f1s in threshold_f1s.items():
    mean_f1 = np.mean(f1s)
    print(f"  {thresh:>10.2f}  {mean_f1:>10.4f}  {np.std(f1s):>8.4f}")
    if mean_f1 > best_f1:
        best_f1, best_thresh = mean_f1, thresh

print(f"\nBest threshold: {best_thresh:.2f} (F1={best_f1:.4f})")



── Optimizing classification threshold (5-fold CV) ──

   Threshold     Mean F1       Std
        0.30      0.9614    0.0014
        0.35      0.9609    0.0013
        0.40      0.9606    0.0014
        0.45      0.9602    0.0014
        0.50      0.9598    0.0014
        0.55      0.9594    0.0013
        0.60      0.9590    0.0013
        0.65      0.9585    0.0014
        0.70      0.9580    0.0014
        0.75      0.9574    0.0012

Best threshold: 0.30 (F1=0.9614)
